# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I am choosing this lane because the core question it answers — *which pages should a content team review first?* — is a direct, actionable decision-support problem, not an abstract modelling exercise. The starter dataset already shows that over half the pages (54.2%) are trending downward, which means a content team cannot review everything; they need a ranked list. The starter pipeline proves the concept works: a random forest improved precision@50 from 0.240 (baseline rules) to 0.740, meaning the top-50 list went from ~12 to ~37 genuine review candidates. My capstone goal is to build a stronger, validated version of this on the full warehouse data, with future-looking labels (prior 90 days → next 30 days decline), proper client-holdout validation, transparent reason codes, and honest caveats about what the ranking can and cannot promise.

In [1]:
# Setup: load starter dataset and confirm basic facts
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Starter dataset: {len(df):,} pages across {df['client_id'].nunique()} pseudonymized clients")
print(f"Columns: {df.shape[1]}")
print(f"Date window: trailing 90-day metrics (pre-aggregated snapshot)")

Starter dataset: 30,000 pages across 32 pseudonymized clients
Columns: 44
Date window: trailing 90-day metrics (pre-aggregated snapshot)


## 2. The question: decision, action, cost of a wrong call

### Research question

> *Given observable search-performance and content signals from the prior 90 days, which pages should a content reviewer examine first for refresh, expansion, or protection — and why?*

### The decision this improves

A content strategist at FlyRank’s client team currently has thousands of pages and limited review capacity. Today, the decision of *which page to look at next* is guided by hand-tuned rule-based scores. My work aims to improve that prioritisation by learning which combinations of signals best predict that a page’s visibility will decline in the near future — so the reviewer spends their limited time on the most promising candidates first.

### Unit of analysis

One **content item** (a pseudonymised page). Each row in the starter dataset is one page with its trailing 90-day metrics.

### The output

A **ranked review queue**: a list of pages ordered by a refresh-opportunity score, each tagged with:
- a numerical score (probability of future decline or opportunity);
- one or more **reason codes** explaining *why* the page ranked high (e.g., `declining_with_demand`, `page_one_decay_risk`, `low_ctr_visible_page`);
- a suggested action category (`refresh`, `expand`, `protect`, `monitor`);
- a confidence label (`high`, `medium`, `low`).

### Who acts on it, and what they do

A **content editor or SEO strategist** at a client team reviews the top-K pages from the queue. For each page, they decide whether to rewrite, expand, update metadata, merge with a related page, or flag for monitoring. The reason codes help them understand *why* the system flagged the page, so they can apply editorial judgement rather than blindly trusting a score.

### Cost of a wrong recommendation

- **False positive** (flagging a healthy page): the reviewer wastes time investigating a page that does not need work. At scale, this erodes trust in the tool and wastes limited editorial capacity.
- **False negative** (missing a declining page): a page that genuinely needs attention continues to lose visibility, clicks, and potential revenue. The longer the decline goes unnoticed, the harder recovery becomes.
- The false-negative cost is arguably higher in most cases, but the practical constraint is reviewer capacity — so **precision@K** ("of the top K pages I surface, how many truly deserve review?") is the metric that best matches the real workflow.

In [2]:
# The framing paragraph (from the framing-ml-problems skill template)
print("""
FRAMING PARAGRAPH
-----------------
For a content strategist deciding which pages to review first,
we will build a ranked refresh-opportunity queue from observable
search-performance and content signals, scoring each page's
likelihood of near-future visibility decline, measured by
precision@K (how many of the top-K flagged pages truly deserve review).

A wrong call costs wasted reviewer hours (false positive) or a
missed decline that compounds over time (false negative).

A plain rule is not enough because the starter baseline's
precision@50 is only 0.240 (12/50 correct), while a learned
model reaches 0.740 (37/50 correct) — the pattern across
20+ interacting signals is too complex for hand-tuned thresholds.

We will claim only observed, directional, and decision-support
results — never causal proof that a refresh caused a recovery.
""")


FRAMING PARAGRAPH
-----------------
For a content strategist deciding which pages to review first,
we will build a ranked refresh-opportunity queue from observable
search-performance and content signals, scoring each page's
likelihood of near-future visibility decline, measured by
precision@K (how many of the top-K flagged pages truly deserve review).

A wrong call costs wasted reviewer hours (false positive) or a
missed decline that compounds over time (false negative).

A plain rule is not enough because the starter baseline's
precision@50 is only 0.240 (12/50 correct), while a learned
model reaches 0.740 (37/50 correct) — the pattern across
20+ interacting signals is too complex for hand-tuned thresholds.

We will claim only observed, directional, and decision-support
results — never causal proof that a refresh caused a recovery.



## 3. Quick look at the data (2–3 real numbers)

Below I load the starter CSV and extract three real numbers that justify spending seven weeks on Lane 2.

In [3]:
# --- NUMBER 1: The scale of the problem ---
# How many pages are trending down, and how many have real demand?

declining = df[df['trend_direction'] == 'down']
declining_with_demand = declining[declining['impressions_90d'] >= 100]

print("NUMBER 1 — The scale of the decline problem")
print("=" * 50)
print(f"Total pages in starter dataset:       {len(df):>8,}")
print(f"Pages trending down:                  {len(declining):>8,}  ({len(declining)/len(df)*100:.1f}%)")
print(f"  … of those, with ≥100 impressions:   {len(declining_with_demand):>8,}")
print(f"  … their median impressions_90d:       {declining_with_demand['impressions_90d'].median():>8,.0f}")
print()
print("Interpretation: Over half (54.2%) of all pages are declining.")
print("13,152 of them have meaningful search demand. No team can manually")
print("review 13,000+ pages — they need a ranked list to focus effort.")

NUMBER 1 — The scale of the decline problem
Total pages in starter dataset:         30,000
Pages trending down:                    16,262  (54.2%)
  … of those, with ≥100 impressions:     13,152
  … their median impressions_90d:          1,620

Interpretation: Over half (54.2%) of all pages are declining.
13,152 of them have meaningful search demand. No team can manually
review 13,000+ pages — they need a ranked list to focus effort.


In [4]:
# --- NUMBER 2: ML already beats rules on this data ---
# The starter pipeline's model results (from outputs/model_report.md)

print("NUMBER 2 — ML already outperforms hand-tuned rules")
print("=" * 50)
print("From the starter pipeline (client-holdout validation):")
print()
print(f"  {'Method':<25} {'ROC AUC':>9} {'Avg Prec':>9} {'Prec@50':>9}")
print(f"  {'-'*25} {'-'*9} {'-'*9} {'-'*9}")
print(f"  {'Baseline rules':<25} {'0.627':>9} {'0.468':>9} {'0.240':>9}")
print(f"  {'Logistic regression':<25} {'0.700':>9} {'0.522':>9} {'0.400':>9}")
print(f"  {'Decision tree':<25} {'0.742':>9} {'0.575':>9} {'0.540':>9}")
print(f"  {'Random forest':<25} {'0.750':>9} {'0.618':>9} {'0.740':>9}")
print()
print("Interpretation: The random forest gets 37 of its top 50 right")
print("(precision@50 = 0.740), versus only 12/50 for the baseline rules.")
print("That is a 3x improvement in the metric that matches the real")
print("workflow: 'of the pages I surface first, how many truly need review?'")
print()
print("This result is on a 30k-row starter slice with a proxy label")
print("(current trend_direction, not a future outcome). The capstone")
print("opportunity is to test whether this holds with a stronger,")
print("future-looking label on the full ~79M-row warehouse.")

NUMBER 2 — ML already outperforms hand-tuned rules
From the starter pipeline (client-holdout validation):

  Method                      ROC AUC  Avg Prec   Prec@50
  ------------------------- --------- --------- ---------
  Baseline rules                0.627     0.468     0.240
  Logistic regression           0.700     0.522     0.400
  Decision tree                 0.742     0.575     0.540
  Random forest                 0.750     0.618     0.740

Interpretation: The random forest gets 37 of its top 50 right
(precision@50 = 0.740), versus only 12/50 for the baseline rules.
That is a 3x improvement in the metric that matches the real
workflow: 'of the pages I surface first, how many truly need review?'

This result is on a 30k-row starter slice with a proxy label
(current trend_direction, not a future outcome). The capstone
opportunity is to test whether this holds with a stronger,
future-looking label on the full ~79M-row warehouse.


In [5]:
# --- NUMBER 3: Page-1 declining pages are high-value review candidates ---

page1_declining = declining[declining['position_tier'].isin(['top_3', 'page_1'])]

print("NUMBER 3 — High-value candidates exist and are identifiable")
print("=" * 50)
print(f"Declining pages currently on Google page 1:  {len(page1_declining):,}")
print(f"  Their median 90-day impressions:            {page1_declining['impressions_90d'].median():,.0f}")
print(f"  Their median avg_position:                  {page1_declining[page1_declining['avg_position']>0]['avg_position'].median():.1f}")
print()
print("Interpretation: 7,289 pages are declining while still ranking")
print("on page 1 of search results. These are the highest-value")
print("candidates — they have real visibility to protect, and a")
print("reviewer acting on them early could prevent further loss.")
print("But 7,289 is still too many for a manual pass; a ranked")
print("queue that puts the most at-risk pages first is essential.")

NUMBER 3 — High-value candidates exist and are identifiable
Declining pages currently on Google page 1:  7,289
  Their median 90-day impressions:            1,071
  Their median avg_position:                  6.4

Interpretation: 7,289 pages are declining while still ranking
on page 1 of search results. These are the highest-value
candidates — they have real visibility to protect, and a
reviewer acting on them early could prevent further loss.
But 7,289 is still too many for a manual pass; a ranked
queue that puts the most at-risk pages first is essential.


In [6]:
# --- Summary table ---
print("\nSUMMARY: Three numbers that justify Lane 2")
print("=" * 55)
print(f"  1. 54.2% of pages are declining — too many to review manually")
print(f"  2. ML precision@50 = 0.740 vs rules 0.240 — 3x improvement")
print(f"  3. 7,289 page-1 decliners exist — high-value, needs ranking")
print()
print("These numbers show the problem is real, the data has signal,")
print("and a learned model can meaningfully outperform a rule-based")
print("approach at surfacing the right pages for human review.")


SUMMARY: Three numbers that justify Lane 2
  1. 54.2% of pages are declining — too many to review manually
  2. ML precision@50 = 0.740 vs rules 0.240 — 3x improvement
  3. 7,289 page-1 decliners exist — high-value, needs ranking

These numbers show the problem is real, the data has signal,
and a learned model can meaningfully outperform a rule-based
approach at surfacing the right pages for human review.


## 4. Careful words: what I can and can’t claim

### What this work CAN claim (with proper validation)

- **Observed associations:** “We observed that pages with these signal combinations were more likely to be trending downward.”
- **Directional evidence:** “The model’s top-ranked pages are more likely to be genuinely declining than the baseline’s top-ranked pages, as measured by precision@K on held-out clients.”
- **Decision-support value:** “Using the model’s ranked queue, a reviewer examining the top 50 pages finds roughly 3x more genuine review candidates than the rule-based baseline.”
- **Relative improvement:** “The learned model improves prioritisation over transparent hand-tuned rules on this dataset.”

### What this work CANNOT claim

- **Causal proof:** I cannot claim that refreshing a flagged page *caused* it to recover. That would require an experiment (A/B test or similar causal design), which this observational data alone cannot provide.
- **Google algorithm factors:** I cannot claim that any signal “ranks pages higher in Google.” The data shows correlations in search visibility, not Google’s internal ranking logic.
- **Guaranteed outcomes:** A high score means “worth reviewing first,” not “will definitely benefit from a refresh.”
- **Generalization beyond the data:** Results validated on 30k starter rows (or even the full 79M-row warehouse) apply to these clients and this time window. Extending to new clients or future periods requires further validation.
- **AI-referral predictions:** AI-session data is sparse (1,930 of 30,000 pages have any AI sessions). I will not train a classifier on this signal alone.

### Language I will use

| Instead of… | I will write… |
|---|---|
| “This page needs a refresh” | “This page is a review candidate based on observed decline signals” |
| “The model predicts recovery” | “The model identifies pages with a higher likelihood of continued decline” |
| “Word count causes ranking” | “Word count is associated with differences in visibility in this dataset” |
| “Proven to work” | “Outperformed the baseline on held-out clients in this snapshot” |

In [7]:
# Why this is NOT just "train a model"
print("WHY THIS IS NOT JUST 'TRAIN A MODEL'")
print("=" * 50)
print("""
This project is about improving a DECISION, not achieving a high AUC.

The decision: which pages should a content reviewer look at first?
The action:   a human editor reviews, rewrites, expands, or protects.
The metric:   precision@K — does the top of the list actually help?
The baseline: transparent hand-tuned rules that already exist.
The standard: the model must EARN its complexity by beating the
              baseline on held-out clients, with honest validation.

If the model does not beat the baseline, the baseline wins — and
that is a valid, publishable result. The goal is better decisions,
not a fancier model.

The capstone adds:
  - Future-looking labels (prior 90d features → next 30d outcome)
  - Strict leakage audit (feature window never overlaps target window)
  - Client-holdout validation (test on clients the model never saw)
  - Reason codes (every recommendation comes with a human-readable why)
  - Careful language (observed, directional, decision-support only)
""")

WHY THIS IS NOT JUST 'TRAIN A MODEL'

This project is about improving a DECISION, not achieving a high AUC.

The decision: which pages should a content reviewer look at first?
The action:   a human editor reviews, rewrites, expands, or protects.
The metric:   precision@K — does the top of the list actually help?
The baseline: transparent hand-tuned rules that already exist.
The standard: the model must EARN its complexity by beating the
              baseline on held-out clients, with honest validation.

If the model does not beat the baseline, the baseline wins — and
that is a valid, publishable result. The goal is better decisions,
not a fancier model.

The capstone adds:
  - Future-looking labels (prior 90d features → next 30d outcome)
  - Strict leakage audit (feature window never overlaps target window)
  - Client-holdout validation (test on clients the model never saw)
  - Reason codes (every recommendation comes with a human-readable why)
  - Careful language (observed, directio

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.